<a href="https://colab.research.google.com/github/Ezequiel-Cley/01-APRENDIZADOS/blob/main/Desafio_Dio_Assistente_de_Voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Gravação de Voz no Google Colab (Python) e JavaScript

In [24]:
# Bibliotecas necessárias para gravação de Audio no Google COlab
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

In [41]:
language = 'pt' # Definido a linguagem para usar na transcrição e criação de audio

In [25]:
# Comando em texto de JavaScript para gravação de voz
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true})
  recorder = new MediaRecorder(stream)
  chunks  = []
  recorder.ondataavailable = e => chunks .push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

In [26]:
# Função resposável por executar o Código em JavaScript passar os parametros e realizar a gravação da voz
def record(sec=5):
  print('Iniciado Gravação!!!')
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (sec * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

In [43]:
record_file = record()
display(Audio(record_file, autoplay=True))

Iniciado Gravação!!!


<IPython.core.display.Javascript object>

###Iniciando o reconhecimento de voz com Whisper

In [5]:
# Instalando biblioteca whisper
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [44]:
import whisper # Importando biblioteca para reconhecimento de voz

# Criando o modelo pequeno
model = whisper.load_model("small")

# Chamando o modelo para fazer transcrição
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Você conhece o canal no YouTube chamado Data Insights Cursos?


### Conexão com API da Gemini

No projeto apresentando pela dio pediu para usar o openai (ChatGPT) porém como não conseguir usar de forma gratuita, migrei para o Gemini para poder usar ela de forma gratuita.

In [14]:
pip install -q -U google-genai # Baixando a biblioteca para conectividade com API do Gemini

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.7/724.7 kB 8.3 MB/s eta 0:00:00


In [45]:
from google import genai # Importando Biblioteca

# Chave da API para conexão
GEMINI_API_KEY = 'GEMINI_API_KEY'

# Definido client para conectar com Gemini
client = genai.Client(api_key=GEMINI_API_KEY)

# Chamando o modelo para gerar as informações desejadas da pergunta em audio.
response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=transcription
)

# Armazenado a resposta em texto
gemini_response = response.text

print(response.text)

Sim, conheço! O canal **Data Insights Cursos** é uma das referências no Brasil quando o assunto é **Business Intelligence (BI)**, **Análise de Dados** e ferramentas da Microsoft, como **Power BI** e **Excel**.

Aqui estão alguns pontos principais sobre o canal e a escola por trás dele:

1.  **Fundadores:** O canal é liderado por **Karine Lago** (que é Microsoft MVP - Most Valuable Professional) e **David Elias**. Ambos são profissionais muito respeitados na comunidade de dados.
2.  **Foco do Conteúdo:**
    *   **Power BI:** Eles têm conteúdos que vão desde o básico até o nível muito avançado, com foco especial em **DAX** (a linguagem de fórmulas do Power BI) e **Power Query** (M).
    *   **Excel:** Tutoriais sobre fórmulas avançadas, Power Pivot e automação.
    *   **Design de Dashboards:** Eles dão muita importância à parte visual e à experiência do usuário (UX/UI aplicada a dados), ensinando como criar relatórios que sejam funcionais e bonitos.
    *   **SQL e Carreira:** Também a

###Sitentizado a resposta com o gTTS

In [13]:
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [32]:
from gtts import gTTS # Importando bibliotecas

In [46]:
# Passado o texto para converter em audio a resposta
gtts_object = gTTS(text=gemini_response, lang=language, slow=False)

# definido o local onde devo salvar o arquivo de audio
response_audio = "/content/response_audio.wav"

# Salvando o arquivo de audio
gtts_object.save(response_audio)

# Criando um display para ouvir o audio
display(Audio(response_audio, autoplay=True))